# Genixcure PCR 11개 — Primer 방향·역반복 말단 검증

**입력:** `PCR_11_Reanalysis_QC_v2`의 완료된 샘플별 `single_end_unique.csv.gz` 및 `reanalysis_summary.json`.

**목적:** Brain/Motor/SH 샘플에서 `both_end=0`인 이유를 확인하기 위한 증거 수집. `single_end`는 한쪽 primer 제거 결과이므로 이 파일의 말단 빈도는 이미 일부 trimming의 영향을 받습니다. 이 분석은 원본 FASTQ에서의 primer 진단을 대체하지 않으며, motif만으로 hairpin/concatemer/서열 다양성을 확정하지 않습니다.

**작업:** 완료된 샘플 자동 인식 → read_count 가중 5′/3′ 상위 20-mer 계산 → 역상보 말단 쌍 확인 → 주요 결과 CSV 생성. 기존 결과 보존, 추가 trimming 없음.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from collections import Counter
import gzip,csv,json, pandas as pd
ROOT=Path('/content/drive/MyDrive/Genixcure_PCR_Merge_Analysis/PCR_11_Reanalysis_QC_v2')
OUT=ROOT/'Primer_Diagnostic_v1'
OUT.mkdir(exist_ok=True)
SAMPLES=['PCR_6_13','Brain_1','Brain_2','Brain_3','Motor_1','Motor_2','Motor_3','SH_SY5Y_1','SH_SY5Y_2','SH_SY5Y_3','Input']
K=20
TOP=10
TRANS=str.maketrans('ACGT','TGCA')
def rc(s):return s.translate(TRANS)[::-1]


In [ ]:
def analyze(sample):
    folder=ROOT/sample
    src=folder/(sample+'_single_end_unique.csv.gz')
    summary=folder/'reanalysis_summary.json'
    if not src.exists() or not summary.exists():
        return {'sample':sample,'status':'NOT_COMPLETE'},[]
    qc=json.loads(summary.read_text())
    h,t=Counter(),Counter()
    reads=unique=0
    with gzip.open(src,'rt',newline='') as f:
        rd=csv.DictReader(f)
        assert rd.fieldnames==['sequence','read_count','length_bp']
        for row in rd:
            seq=row['sequence'].upper();count=int(row['read_count']);length=int(row['length_bp'])
            if count<=0 or len(seq)!=length:raise ValueError('Invalid input: '+sample)
            unique+=1;reads+=count
            if length>=K:
                h[seq[:K]]+=count
                t[seq[-K:]]+=count
    assert reads==qc['category_counts']['single_end']['reads'],(sample,reads)
    assert unique==qc['category_counts']['single_end']['unique'],(sample,unique)
    best_head=h.most_common(TOP);best_tail=t.most_common(TOP)
    records=[]
    for label,data,other in [('five_prime',best_head,t),('three_prime',best_tail,h)]:
        for rank,(motif,count) in enumerate(data,1):
            records.append({'sample':sample,'end':label,'rank':rank,'motif':motif,
                            'weighted_reads':count,'fraction_single_end_reads':count/reads if reads else 0,
                            'reverse_complement':rc(motif),'rc_opposite_end_reads':other.get(rc(motif),0)})
    top_motif,top_count=best_head[0] if best_head else ('',0)
    return {'sample':sample,'status':'OK','single_end_reads':reads,
            'single_end_unique':unique,'five_prime_top20':top_motif,
            'five_prime_top20_pct':round(100*top_count/reads,3) if reads else 0,
            'reverse_complement_at_three_prime_reads':t.get(rc(top_motif),0),
            'reverse_complement_at_three_prime_pct':round(100*t.get(rc(top_motif),0)/reads,3) if reads else 0,
            'both_end_reads':qc['category_counts']['both_end']['reads']},records

rows=[];motifs=[]
for sample in SAMPLES:
    result,record=analyze(sample)
    rows.append(result);motifs.extend(record)
    print(sample,result['status'],result.get('five_prime_top20',''),
          result.get('five_prime_top20_pct',''),flush=True)

summary=pd.DataFrame(rows)
summary.to_csv(OUT/'primer_terminal_diagnostic_summary.csv',index=False)
pd.DataFrame(motifs).to_csv(OUT/'primer_terminal_top_motifs.csv',index=False)
display(summary)
print('Saved:',OUT)


## 해석 시 주의사항

- 높은 역상보 20-mer 빈도는 **서열 양끝의 패턴**을 뜻하며 정확히 같은 분자 내 inverted repeat라는 증명은 아닙니다. 이 표는 양끝의 주변부 분포를 따로 합친 것이라 같은 Read에서의 동시 출현율을 의미하지 않습니다.
- `both_end=0`은 알려진 AAV F/R 프라이머 **완전 일치** 판정 실패이며, 다른 라이브러리 구조·변이·말단 길이·오리엔테이션을 의심할 근거입니다. 저품질이라는 판정은 아닙니다.
- 원시 FASTQ에서 *untrimmed* 말단 motif와 adapter/primer 위치를 확인한 뒤에만 primer 패널이나 trimming 규칙을 수정하세요.
- 이 notebook은 재분석 실행이나 백업 삭제를 하지 않습니다.